# Notebook 2 — Optimised CNN (CIFAR DecaLuminarNet) on CIFAR-10
**Deep Learning | Computer Vision | 2025**

Systematically enhance the baseline CNN through five targeted improvements,
culminating in the custom **CIFAR DecaLuminarNet** architecture.

| Enhancement | Technique |
|---|---|
| Stable training | BatchNormalization after every Conv |
| Regularisation | Dropout (0.25 → 0.5) |
| Richer features | 4 deeper convolutional blocks (2–3 Conv each) |
| Better generalisation | Data Augmentation (flip, shift, rotate, zoom) |
| Smarter LR | ReduceLROnPlateau + EarlyStopping |

| Model | Accuracy |
|---|---|
| Baseline CNN (NB1) | ~65% |
| **CIFAR DecaLuminarNet (this notebook)** | **~82%** |

> ✅ Open in **Google Colab** → Runtime → Change runtime type → **GPU**


## 1. Imports & Setup

In [ ]:
from time import time
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import SGD, RMSprop, Adagrad, Adadelta, Adam, Adamax, Nadam
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Load & Preprocess

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

train_images_n = train_images.astype('float32') / 255.0
test_images_n  = test_images.astype('float32')  / 255.0
train_labels_enc = keras.utils.to_categorical(train_labels, 10)
test_labels_enc  = keras.utils.to_categorical(test_labels,  10)
y_true = test_labels.flatten()

print(f"Train: {train_images_n.shape}  Test: {test_images_n.shape}")

## 3. Data Augmentation

Artificially expands the training set with random transformations —
forces the model to learn invariant, generalisable features.


In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,        # random ±15° rotation
    width_shift_range=0.1,    # horizontal shift ±10%
    height_shift_range=0.1,   # vertical shift ±10%
    horizontal_flip=True,     # random left-right flip
    zoom_range=0.1,           # zoom ±10%
    fill_mode='nearest'
)
datagen.fit(train_images_n)
print("Augmentation pipeline ready.")

In [ ]:
# Visualise augmentation
sample = train_images_n[np.where(train_labels.flatten()==0)[0][0:1]]
fig, axes = plt.subplots(2, 6, figsize=(16,6))
fig.suptitle('Data Augmentation — Original vs Augmented', fontsize=13, fontweight='bold')
axes[0,0].imshow(sample[0]); axes[0,0].set_title('Original',fontweight='bold'); axes[0,0].axis('off')
axes[1,0].imshow(sample[0]); axes[1,0].axis('off')
gen = datagen.flow(sample, batch_size=1)
for col in range(1,6):
    axes[0,col].imshow(np.clip(next(gen)[0],0,1)); axes[0,col].set_title(f'Aug {col}'); axes[0,col].axis('off')
    axes[1,col].imshow(np.clip(next(gen)[0],0,1)); axes[1,col].axis('off')
plt.tight_layout()
plt.savefig('02_augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Shared Evaluation Helper

In [ ]:
def evaluate_model(model, test_imgs, test_lbls_enc, y_true, label='Model'):
    """Evaluate model and print full metrics + confusion matrix."""
    test_loss, test_acc = model.evaluate(test_imgs, test_lbls_enc, verbose=0)
    y_proba = model.predict(test_imgs, verbose=0)
    y_pred  = np.argmax(y_proba, axis=1)

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"  Test Accuracy  : {test_acc*100:.2f}%")
    print(f"  Test Loss      : {test_loss:.4f}")
    print(f"  Precision      : {precision_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"  Recall         : {recall_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"  F1-Score       : {f1_score(y_true,y_pred,average='weighted'):.4f}")
    print(f"{'='*55}")
    print("\nPer-Class Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{label} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    fname = label.replace(' ','_').replace('/','_')
    plt.savefig(f'{fname}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    return test_acc, y_pred

## 5. CIFAR DecaLuminarNet Architecture

Custom deep CNN inspired by VGG-style architecture with modern regularisation.

| Block | Filters | Technique |
|---|---|---|
| Block 1 | 64 × 2 Conv | BatchNorm, ELU, Dropout 0.25 |
| Block 2 | 128 × 2 Conv | BatchNorm, ELU, Dropout 0.25 |
| Block 3 | 256 × 3 Conv | BatchNorm, ELU, Dropout 0.25 |
| Block 4 | 512 × 3 Conv | BatchNorm, ELU, Dropout 0.25 |
| Head | Dense(512→512→256→128→64→10) | BatchNorm, Dropout 0.5 |

**`Deca`** = 10 CIFAR-10 classes · **`LuminarNet`** = illuminating deep performance


In [ ]:
def build_decaluminarnet():
    """
    CIFAR DecaLuminarNet — Enhanced CNN for CIFAR-10.
    Achieves ~82%+ accuracy with strong regularisation.
    Architecture inspired by Matthias Bartolo's advanced CNN work.
    """
    model = models.Sequential(name='CIFAR_DecaLuminarNet')

    # ── Block 1 : 64 filters ────────────────────────────────────────────────
    model.add(layers.Conv2D(64,(3,3),activation='elu',padding='same',input_shape=(32,32,3)))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # ── Block 2 : 128 filters ───────────────────────────────────────────────
    model.add(layers.Conv2D(128,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # ── Block 3 : 256 filters (3 conv layers) ───────────────────────────────
    model.add(layers.Conv2D(256,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # ── Block 4 : 512 filters (3 conv layers) ───────────────────────────────
    model.add(layers.Conv2D(512,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(512,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(512,(3,3),activation='elu',padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # ── Classifier Head ─────────────────────────────────────────────────────
    model.add(layers.Flatten())
    model.add(layers.Dense(512,activation='relu')); model.add(layers.BatchNormalization()); model.add(layers.Dropout(0.5))
    model.add(layers.Dense(512,activation='relu')); model.add(layers.BatchNormalization()); model.add(layers.Dropout(0.5))
    model.add(layers.Dense(256,activation='relu')); model.add(layers.BatchNormalization()); model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128,activation='relu')); model.add(layers.BatchNormalization()); model.add(layers.Dropout(0.5))
    model.add(layers.Dense(64, activation='relu')); model.add(layers.BatchNormalization()); model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10))
    model.add(layers.Activation('softmax'))
    return model

decaluminar = build_decaluminarnet()
decaluminar.summary()
print(f"\nTotal parameters: {decaluminar.count_params():,}")

## 6. Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_decaluminarnet.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]
print("Callbacks: EarlyStopping | ReduceLROnPlateau | ModelCheckpoint")

## 7. Compile & Train

In [ ]:
decaluminar.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=3.0),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

BATCH = 128
EPOCHS = 150

s = time()
history = decaluminar.fit(
    datagen.flow(train_images_n, train_labels_enc, batch_size=BATCH),
    steps_per_epoch=len(train_images_n) // BATCH,
    epochs=EPOCHS,
    validation_data=(test_images_n, test_labels_enc),
    callbacks=callbacks,
    verbose=1
)
print(f"Total training time: {round(time()-s,2)}s")

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18,5))
fig.suptitle('CIFAR DecaLuminarNet — Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', lw=2, color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val',   lw=2, color='green', ls='--')
axes[0].plot(history.history['loss'],     label='Train Loss', lw=2, color='purple', ls=':')
axes[0].plot(history.history['val_loss'], label='Val Loss',   lw=2, color='cyan',   ls='-.')
axes[0].set_title('Accuracy & Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', lw=2, color='green')
axes[1].axhline(y=0.82, color='red', ls='--', lw=1.5, label='82% target')
axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

if 'lr' in history.history:
    axes[2].plot(history.history['lr'], color='orange', lw=2)
    axes[2].set_title('Learning Rate (adaptive)'); axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_decaluminarnet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluate DecaLuminarNet

In [ ]:
acc_dln, y_pred_dln = evaluate_model(
    decaluminar, test_images_n, test_labels_enc, y_true,
    label='CIFAR DecaLuminarNet'
)

## 10. Architecture Comparison — Accuracy Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
model_names = ['Baseline CNN\n(2-block, no reg)', 'CIFAR DecaLuminarNet\n(4-block, BN+Drop+Aug)']
accs = [65.0, acc_dln*100]
colors = ['#e74c3c', '#27ae60']
bars = ax.bar(model_names, accs, color=colors, width=0.4, edgecolor='white', lw=2)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.4,
            f'{a:.2f}%', ha='center', fontsize=13, fontweight='bold')
ax.set_ylim(55, 90)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Baseline vs CIFAR DecaLuminarNet', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('02_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Improvement: +{acc_dln*100 - 65.0:.2f} percentage points")

## 11. Summary

| Technique | Effect |
|---|---|
| **ELU activation** | Smoother gradients, better convergence than ReLU |
| **BatchNormalization** | Stable training, acts as implicit regulariser |
| **Dropout (0.25–0.5)** | Closed train/val accuracy gap significantly |
| **4 deeper blocks** | Richer spatial features (64→128→256→512) |
| **Augmentation** | Improved generalisation on unseen data |
| **ReduceLROnPlateau** | Finer weight updates in later epochs |

**Final: ~82% (+17% over baseline)**

➡️ **Notebook 3** applies MobileNetV2 Transfer Learning → **~88%**
